# Week 4: Evaluation


## Generating Ground Truth Data

In [4]:
from ingest import load_faq_data
documents = load_faq_data()

In [5]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

103

In [6]:
documents = documents_llm

doc = documents[0]
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [7]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [8]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [9]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [10]:
import json

user_prompt = json.dumps(doc)

In [11]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [12]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [13]:
result = response.output_parsed

print(result)

questions=['I just found this course late — can I still join and start from here?', 'Is it okay to enroll now if I missed the start date?', 'If I join the course after it already started, can I still take part?', 'Do late joiners still have a chance to get a certificate in this course?', 'What’s the deadline for submitting the project if I want to be eligible for the certificate?']


In [14]:
from evaluation_utils import llm_structured

In [15]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)


['Can I join the course late if I just found it?', 'Is it still okay to start the course now even though it already began?', 'If I sign up after the course started, can I still get a certificate?', 'Do I need to finish and submit the project before submissions close to earn a certificate?', 'What happens if I join now but miss the project submission deadline for the certificate?']


In [16]:
usage.input_tokens, usage.output_tokens

(207, 92)

In [17]:
from evaluation_utils import calc_price

In [18]:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525, 'output_cost': 0.000414, 'total_cost': 0.00056925}

In [19]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I join the course late if I just found it?',
  'document': '74eb249bbf'},
 {'question': 'Is it still okay to start the course now even though it already began?',
  'document': '74eb249bbf'},
 {'question': 'If I sign up after the course started, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to finish and submit the project before submissions close to earn a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What happens if I join now but miss the project submission deadline for the certificate?',
  'document': '74eb249bbf'}]

## Generating Ground Truth for All Documents

In [20]:
from evaluation_utils import llm_structured_retry

In [21]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [22]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [24]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
# with ThreadPoolExecutor(max_workers=6) as pool:
#     results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/103 [00:00<?, ?it/s]

In [26]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

515

In [27]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.07650975000000002

In [28]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.07650975000000002

In [29]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [31]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)